# Football Intelligence Scouting Platform — EDA & Modeling

This notebook walks through the analytical core of the platform: loading data, cleaning, exploratory analysis, feature engineering, model training/evaluation, explainability, and the scoring/recommendation logic.

> The dashboard (`app.py`) reuses the exact same `src/` modules, so this notebook is a faithful, reproducible view of what powers the app.

**Note:** the default dataset is *synthetic sample data*. Drop a real CSV into `data/raw/` to analyze real players.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.data_loader import load_raw_data
from src.preprocessing import clean_data, preprocessing_report
from src.feature_engineering import engineer_features, REGRESSION_FEATURES, CLASSIFICATION_FEATURES
from src.modeling import train_all_models
from src.recommendation import compute_undervalue_score, add_trajectory_category, recommend_players

pd.set_option('display.max_columns', 60)
print('Imports OK')

## 1. Load data (3-tier fallback)

In [ ]:
raw, source = load_raw_data()
print('source:', source, '| shape:', raw.shape)
raw.head()

## 2. Clean & standardize

In [ ]:
clean = clean_data(raw)
print('Cleaning report:', preprocessing_report(raw, clean))
print('Remaining NaNs:', int(clean.isna().sum().sum()))
clean[['name','age','position','position_group','overall_rating','potential','market_value_eur','wage_eur']].head()

## 3. Exploratory Data Analysis

In [ ]:
clean[['age','overall_rating','potential','market_value_eur','wage_eur']].describe().round(0)

In [ ]:
# Value & count by position group
clean.groupby('position_group').agg(
    players=('player_id','size'),
    avg_overall=('overall_rating','mean'),
    avg_value_eur=('market_value_eur','mean'),
).round(0).sort_values('avg_value_eur', ascending=False)

In [ ]:
# Correlation of key drivers with market value
cols = ['age','overall_rating','potential','wage_eur','market_value_eur']
clean[cols].corr().round(2)['market_value_eur'].sort_values(ascending=False)

## 4. Feature engineering & proxy targets
See `docs/methodology.md` for the rationale behind the proxy targets.

In [ ]:
feats = engineer_features(clean)
print('Engineered shape:', feats.shape)
print('Transfer-success class balance:', round(feats['transfer_success'].mean(), 3))
feats[['name','potential_gap','years_to_peak','performance_index','rating_per_wage','future_value_eur','value_growth_pct','transfer_success']].head()

## 5. Train & evaluate models

In [ ]:
bundle = train_all_models(feats)
print('Regression  (future value):', {k: (round(v,3) if isinstance(v,float) else v) for k,v in bundle.regression_metrics.items()})
print('Classification (transfer):', {k: (round(v,3) if isinstance(v,float) else v) for k,v in bundle.classification_metrics.items()})

In [ ]:
print('Top value-model drivers:')
display(bundle.regression_importance.head(6))
print('Top transfer-model drivers:')
display(bundle.classification_importance.head(6))

## 6. Undervalue scoring & hidden talents

In [ ]:
data = compute_undervalue_score(bundle.predictions)
data = add_trajectory_category(data)
print('Trajectory distribution:', data['trajectory_category'].value_counts().to_dict())

top_talents = data.sort_values('undervalue_score', ascending=False).head(10)
top_talents[['name','position','age','overall_rating','potential','market_value_eur','predicted_future_value_eur','predicted_growth_pct','undervalue_score','undervalue_reason']]

## 7. Club Recruitment Engine — example query
A budget-conscious club wants a creative midfielder, age 18–27, potential >= 78.

In [ ]:
shortlist = recommend_players(
    data, position_group='Midfielder', max_budget=40e6,
    age_range=(18, 27), min_potential=78, style='Creative playmaking', top_n=8,
)
shortlist[['name','position','club','age','market_value_eur','predicted_future_value_eur','undervalue_score','fit_score','recommendation_reason']]

## 8. Conclusion

- The pipeline cleanly turns raw player data into four decision tools.
- The value regressor learns a growth-adjusted valuation (R² high because current value is a strong input).
- The transfer classifier reaches a realistic ROC-AUC on a documented proxy label.
- The Undervalue Score and Recruitment Fit Score stay transparent and explainable.

Run the full interactive product with `streamlit run app.py`. See `docs/` for methodology, the data dictionary, and the model card.